# Grad-CAM explainability and failure-case analysis

This notebook presents Grad-CAM comparisons for the six 500-class AlexNet, ResNet18, and ResNet50 checkpoints. It does not modify teammate code, checkpoints, datasets, or previous outputs.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

start = Path.cwd().resolve()
workspace = next((p for p in (start, *start.parents) if (p / 'reproduction').is_dir()), None)
if workspace is None:
    raise RuntimeError('Could not locate the workspace root')
output_dir = workspace / 'reproduction' / 'outputs' / 'deep' / 'gradcam_v2'
if not output_dir.is_dir():
    raise FileNotFoundError(f'Run the Grad-CAM analysis first: {output_dir}')
print('Workspace:', workspace)
print('Grad-CAM output:', output_dir)

## Reproducibility metadata

The expanded formal run uses a fixed seed, scans 1,000 validation images, and selects ten confident correct and ten confident incorrect examples using the pretrained ResNet50.

In [ ]:
metadata = json.loads((output_dir / 'run_metadata.json').read_text(encoding='utf-8'))
display(pd.Series(metadata, name='value').to_frame())

## Prediction summary on the selected explanation cases

**Important:** these six examples are intentionally selected with the pretrained ResNet50. The values below describe agreement on these explanation cases only and must not be reported as unbiased test-set accuracy.

In [ ]:
predictions = pd.read_csv(output_dir / 'predictions.csv')
predictions['correct'] = predictions['correct'].astype(bool)
summary = (predictions.groupby('model', sort=True)
           .agg(samples=('correct', 'size'),
                correct_cases=('correct', 'sum'),
                case_accuracy=('correct', 'mean'),
                mean_confidence=('confidence', 'mean'))
           .reset_index())
display(summary.style.format({'case_accuracy': '{:.1%}', 'mean_confidence': '{:.1%}'}))

## Side-by-side Grad-CAM evidence

Each figure contains the same input image followed by all six models. Titles report the ground truth, prediction, correctness, and confidence.

In [ ]:
figure_paths = sorted(output_dir.glob('sample_*.png'))
if not figure_paths:
    raise FileNotFoundError('No Grad-CAM figures were generated')
for figure_path in figure_paths:
    image = Image.open(figure_path)
    display(image)

## Notable cases for the report

- **Sample 01 - Tringa totanus:** the pretrained ResNet50 localises individual birds despite the wide scene; AlexNet random predicts another bird species with weak confidence.
- **Sample 05 - Abudefduf sexfasciatus:** all six models are correct, and the stronger models emphasise the diagnostic dark body stripes.
- **Sample 11 - Camptostoma imberbe:** the pretrained ResNet50 attends to the bird but confidently predicts Saxicola caprata, supporting a fine-grained bird-species confusion rather than pure background bias.
- **Sample 12 - Aquila heliaca:** the birds occupy a small part of the frame; several models attend to them but confuse the eagle with falcon species, showing sensitivity to object scale and inter-class similarity.
- **Sample 14 - Bombus hypnorum:** pretrained models attend strongly to the bee yet predict Bombus mixtus, while some random-start models attend to flowers or borders. This separates species-level confusion from contextual bias.
- **Sample 19 - Dicksonia squarrosa:** most models focus on the fern crown but confuse it with Cyathea smithii; the error is consistent with similar plant morphology.


## Interpretation guide

For each case, examine whether the strongest activation lies on the organism, on a diagnostic body part, or on background/context. Compare pretrained and randomly initialised variants within the same architecture. Wrong predictions with activation on the organism suggest fine-grained visual confusion; wrong predictions dominated by background suggest contextual bias. Grad-CAM is qualitative evidence and should be discussed alongside quantitative test metrics, not as proof of causal reasoning.

## Re-run command

Run this command from the workspace root to create a new, non-overwriting output directory:

```powershell
.\.9517venv\Scripts\python.exe reproduction\scripts\run_gradcam_analysis.py --checkpoints data\models\deep_checkpoints --dataset data\inat2021\reproduced\sampled_500\val --output reproduction\outputs\deep\gradcam_v2 --scan-limit 300 --examples-per-group 3 --seed 0
```